### SIF retrieval bias vs aerosol optical depth

RT ensemble (`rt_toa_ensemble_*.nc`) → SVD retrieval (`rt_retrieval_*.nc`) → GCHP aerosol column used for each sample (`column_index`, 1-based).
ΔSIF = retrieved − true water-leaving SIF at 678 nm; only converged retrievals (`status == 1`).


In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from scipy import stats

# read rt generated samples
rt_samples_name = "/home/zhe2/FraLab/PACE_redSIF_PACE/surrogate_meas/full_RT_construction/output/rt_toa_ensemble_badcoxmunk.nc"
rt_samples = xr.open_dataset(rt_samples_name)

# read retrieval to be evaluated
retrieval_name = "/home/zhe2/FraLab/PACE_redSIF_PACE/surrogate_meas/full_RT_construction/output/rt_retrieval_bad_coxmunk_O2A_nPC15_npoly3.nc"
retrieval = xr.open_dataset(retrieval_name)

# read aod
aod_name = "/home/zhe2/FraLab/PACE_redSIF_PACE/surrogate_meas/full_RT_construction/output_aerosol_profiles/gchp_ocean_columns_n500.nc"
aod = xr.open_dataset(aod_name)

UNITS = r"W m$^{-2}$ sr$^{-1}$ µm$^{-1}$"


In [ ]:
# from rt_samples get 1-based aod profile index
column_index = rt_samples.column_index.values
# get ground truth SIF 678
sif_678_true_sample = rt_samples.sif_678.values

# from retrieval get SIF 678 (true and ret, where `true` is for ground truth comparison)
sif_678_true = retrieval.sif_678_true.values
sif_678_ret = retrieval.sif_678_ret.values
status = retrieval.status.values          # 1 = converged

# a sanity check - make sure the ground truth are matching (both stored as float32)
assert sif_678_true.shape == sif_678_true_sample.shape
assert np.allclose(sif_678_true_sample, sif_678_true)
assert column_index.min() >= 1, "aerosols were disabled in this ensemble (column_index = 0)"

ok = status == 1
delta_sif = sif_678_ret - sif_678_true    # retrieved − true
print(f"samples: {ok.size}, converged: {ok.sum()}  (status counts: "
      f"{dict(zip(*np.unique(status, return_counts=True)))})")


In [ ]:
# from column index get aod profiles (GCHP arrays are (lev, sample); lev runs surface → TOA)
icol = column_index - 1
aod_layer = aod.AOD_total_layer.values[:, icol]           # (lev, n_samples)
ss_layer = aod.AOD_seasalt.values[:, icol]
p_mid = aod.p_mid.values[:, icol]                          # hPa

aod_total = aod.tau_ref.values[icol]                       # column AOD at 550 nm
aod_seasalt = ss_layer.sum(axis=0)
ss_fraction = aod.ss_fraction.values[icol]
aod_species = {name: aod[f"AOD_GCHP_{name}"].values[:, icol].sum(axis=0)
               for name in ["SO4", "SALA", "SALC", "OCPI", "BCPI"]}
aod_species["DUST"] = sum(aod[f"AOD_GCHP_DUST{i}"].values[:, icol].sum(axis=0) for i in range(1, 8))
print(f"total AOD(550) over samples: median {np.median(aod_total):.3f}, "
      f"5–95% [{np.percentile(aod_total, 5):.3f}, {np.percentile(aod_total, 95):.3f}]")

# visualize total AOD and sea salt AOD, and some vertical distributions every 50 profiles
fig, axes = plt.subplots(1, 3, figsize=(15, 4.3), layout="constrained")
bins = np.linspace(0, np.percentile(aod_total, 99.5), 40)
axes[0].hist(aod_total, bins=bins, color="0.55", label="total")
axes[0].hist(aod_seasalt, bins=bins, color="#4C72B0", alpha=0.7, label="sea salt")
axes[0].set_xlabel("Column AOD at 550 nm")
axes[0].set_ylabel("Samples")
axes[0].set_title("(a) Column AOD over the ensemble", loc="left")
axes[0].legend(frameon=False)

names = list(aod_species)
axes[1].boxplot([aod_species[n] for n in names], tick_labels=names, showfliers=False)
axes[1].set_yscale("log")
axes[1].set_ylabel("Column AOD at 550 nm")
axes[1].set_title("(b) AOD by species (box: 25–75%, whiskers 1.5 IQR)", loc="left")

for i in range(0, len(icol), 50):
    axes[2].plot(aod_layer[:, i], p_mid[:, i], lw=1, alpha=0.8,
                 label=f"sample {i + 1} (col {column_index[i]}, τ={aod_total[i]:.2f})")
axes[2].set_ylim(1050, 300)
axes[2].set_xlabel("Layer AOD at 550 nm")
axes[2].set_ylabel("Pressure (hPa)")
axes[2].set_title("(c) Vertical AOD profiles, every 50th sample", loc="left")
axes[2].legend(fontsize=6.5, frameon=False, loc="lower right")
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.show()


In [ ]:
# histogram of delta_SIF (converged retrievals only)
d = delta_sif[ok]
rel = d / np.where(np.abs(sif_678_true[ok]) > 0.05, sif_678_true[ok], np.nan) * 100   # % for SIF > 0.05
print(f"ΔSIF = SIF_ret − SIF_true (n={d.size}): mean {d.mean():+.4f}, median {np.median(d):+.4f}, "
      f"SD {d.std():.4f}, RMSE {np.sqrt(np.mean(d**2)):.4f}  [{UNITS.replace('$', '')}]")
print(f"relative (true SIF > 0.05): median {np.nanmedian(rel):+.1f} %, IQR "
      f"[{np.nanpercentile(rel, 25):+.1f}, {np.nanpercentile(rel, 75):+.1f}] %")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), layout="constrained")
lo, hi = np.percentile(d, [0.5, 99.5])
axes[0].hist(d, bins=np.linspace(lo, hi, 50), color="#0B7A75")
axes[0].axvline(0, color="k", lw=1)
axes[0].axvline(np.median(d), color="#C44E52", lw=1.5, ls="--", label=f"median {np.median(d):+.3f}")
axes[0].set_xlabel(f"ΔSIF$_{{678}}$ = retrieved − true ({UNITS})")
axes[0].set_ylabel("Samples")
axes[0].set_title(f"(a) Absolute error, n = {d.size} converged", loc="left")
axes[0].legend(frameon=False)

r = rel[np.isfinite(rel)]
lo, hi = np.percentile(r, [1, 99])
axes[1].hist(r, bins=np.linspace(lo, hi, 50), color="#4C72B0")
axes[1].axvline(0, color="k", lw=1)
axes[1].set_xlabel("ΔSIF / SIF$_{true}$ (%)")
axes[1].set_title(f"(b) Relative error, true SIF > 0.05 (n = {r.size})", loc="left")
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.show()


In [ ]:
# scatterplot of delta SIF against AODs (converged retrievals), with binned medians
def binned_median(x, y, edges, min_n=15):
    k = np.digitize(x, edges) - 1
    mid, med, q1, q3 = [], [], [], []
    for i in range(len(edges) - 1):
        m = k == i
        if m.sum() >= min_n:
            mid.append(np.median(x[m]))
            q = np.percentile(y[m], [25, 50, 75])
            q1.append(q[0]); med.append(q[1]); q3.append(q[2])
    return map(np.asarray, (mid, med, q1, q3))

predictors = [
    ("Total AOD at 550 nm", aod_total),
    ("Sea-salt AOD at 550 nm", aod_seasalt),
    ("Sea-salt fraction of AOD", ss_fraction),
]
y = delta_sif[ok]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), sharey=True, layout="constrained")
for ax, (name, x_all), letter in zip(axes, predictors, "abc"):
    x = x_all[ok]
    sc = ax.scatter(x, y, s=10, c=sif_678_true[ok], cmap="viridis", alpha=0.6, linewidths=0)
    edges = np.quantile(x, np.linspace(0, 1, 11))           # deciles → equal counts per bin
    mid, med, q1, q3 = binned_median(x, y, edges)
    ax.fill_between(mid, q1, q3, color="#C44E52", alpha=0.2, linewidth=0)
    ax.plot(mid, med, "o-", color="#C44E52", lw=1.8, ms=4, label="binned median (IQR)")
    ax.axhline(0, color="k", lw=0.8)
    rho, p = stats.spearmanr(x, y)
    ax.set_title(f"({letter}) Spearman ρ = {rho:+.2f} (p = {p:.1g})", loc="left")
    ax.set_xlabel(name)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel(f"ΔSIF$_{{678}}$ = retrieved − true ({UNITS})")
lo, hi = np.percentile(y, [1, 99])
axes[0].set_ylim(lo - 0.2 * (hi - lo), hi + 0.2 * (hi - lo))
axes[0].legend(frameon=False, loc="upper right")
fig.colorbar(sc, ax=axes, label=f"True SIF$_{{678}}$ ({UNITS})", shrink=0.9)
plt.show()

# does the AOD dependence survive once geometry is accounted for? (glint geometries are
# unreliable in this bad-Cox-Munk ensemble)
vaz = rt_samples.vaz.values[ok]
sza = rt_samples.sza.values[ok]
vza = rt_samples.vza.values[ok]
for name, x_all in predictors[:1]:
    x = x_all[ok]
    for lab, m in (("backscatter side (vaz > 90°)", vaz > 90), ("forward / glint side (vaz ≤ 90°)", vaz <= 90)):
        rho, p = stats.spearmanr(x[m], y[m])
        print(f"{name}, {lab:34s} n={m.sum():4d}  Spearman ρ = {rho:+.2f} (p = {p:.1g})")
